In [ ]:
"""
K-ELECTRIC DPO TOOLKIT — Complete Python replacement for all Excel DPO tasks
Usage: python dpo_toolkit.py
Just change the INPUT_FILE path and run. Outputs a formatted Excel report.
"""

import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime
import os, sys

# ══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════
INPUT_FILE  = "DPO_Practice_Data.xlsx"
OUTPUT_FILE = f"DPO_Report_{datetime.today().strftime('%Y-%m-%d')}.xlsx"

# number formats — applied to cells so Excel treats them as real numbers
FMT_PCT   = "0.0%"        # 0.416 → shows as 41.6%
FMT_PCT2  = "0.00%"       # 0.2076 → shows as 20.76%
FMT_NUM   = "#,##0"       # 2055086 → shows as 2,055,086
FMT_INT   = "0"           # plain integer

# ══════════════════════════════════════════════════════════════════
# COLOURS
# ══════════════════════════════════════════════════════════════════
BLUE   = "1F4E79";  LBLUE  = "D9E1F2"
GREEN  = "1E5631";  LGREEN = "E2EFDA"
RED    = "C00000";  LRED   = "FFCCCC"
ORANGE = "7B3F00";  LORANGE= "FCE4D6"
PURPLE = "4B0082";  LPURPLE= "EAD1DC"
YELLOW = "FFFF00";  WHITE  = "FFFFFF"
LGRAY  = "F2F2F2"

# ══════════════════════════════════════════════════════════════════
# STYLE HELPERS
# ══════════════════════════════════════════════════════════════════
def hdr(cell, bg=BLUE, fg=WHITE, sz=11, bold=True):
    cell.font      = Font(name="Arial", bold=bold, color=fg, size=sz)
    cell.fill      = PatternFill("solid", start_color=bg)
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

def dat(cell, bg=None, bold=False, center=False, fmt=None):
    cell.font      = Font(name="Arial", size=10, bold=bold)
    cell.alignment = Alignment(horizontal="center" if center else "left", vertical="center")
    if bg:
        cell.fill = PatternFill("solid", start_color=bg)
    if fmt:
        cell.number_format = fmt

def borders(ws, r1, r2, c1, c2):
    t = Side(style="thin")
    b = Border(left=t, right=t, top=t, bottom=t)
    for row in ws.iter_rows(min_row=r1, max_row=r2, min_col=c1, max_col=c2):
        for cell in row:
            cell.border = b

def auto_col(ws, pad=4):
    for col in ws.columns:
        mx = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(mx + pad, 45)

def title(ws, text, ncols, bg=BLUE, row=1):
    ws.merge_cells(f"A{row}:{get_column_letter(ncols)}{row}")
    ws[f"A{row}"] = text
    hdr(ws[f"A{row}"], bg=bg, sz=13)
    ws.row_dimensions[row].height = 28

# ══════════════════════════════════════════════════════════════════
# write_df  — writes DataFrame with proper number types + formats
# col_fmts  — dict {col_name: format_string} for numeric columns
# row_colors — list of (hex_bg) per data row, or None
# ══════════════════════════════════════════════════════════════════
def write_df(ws, df, start_row=3, hdr_bg=BLUE, col_fmts=None, row_bgs=None):
    col_fmts = col_fmts or {}
    # header row
    hr = start_row - 1
    for ci, col in enumerate(df.columns, 1):
        hdr(ws.cell(row=hr, column=ci, value=col), bg=hdr_bg)
    # data rows
    for ri, (_, row_data) in enumerate(df.iterrows(), start_row):
        bg = row_bgs[ri - start_row] if row_bgs else (LGRAY if (ri - start_row) % 2 == 1 else None)
        for ci, col in enumerate(df.columns, 1):
            val = row_data[col]
            # keep NaN out of cells
            if pd.isna(val) if not isinstance(val, str) else False:
                val = ""
            cell = ws.cell(row=ri, column=ci, value=val)
            fmt  = col_fmts.get(col)
            dat(cell, bg=bg, center=True, fmt=fmt)
    borders(ws, hr, start_row + len(df) - 1, 1, len(df.columns))
    auto_col(ws)

# ══════════════════════════════════════════════════════════════════
# summary_block — writes a labelled summary table with real numbers
# items = list of (label, value, optional_format)
# ══════════════════════════════════════════════════════════════════
def summary_block(ws, items, start_row, bg_title, title_text):
    # title cell
    tc = ws.cell(row=start_row, column=1, value=title_text)
    tc.font = Font(name="Arial", bold=True, size=11, color=WHITE)
    tc.fill = PatternFill("solid", start_color=bg_title)
    tc.alignment = Alignment(vertical="center", horizontal="left")
    ws.merge_cells(f"A{start_row}:B{start_row}")

    for i, item in enumerate(items, start_row + 1):
        label = item[0]
        val   = item[1]
        fmt   = item[2] if len(item) == 3 else None

        lc = ws.cell(row=i, column=1, value=label)
        vc = ws.cell(row=i, column=2, value=val)

        lc.font = Font(name="Arial", bold=True, size=10)
        lc.fill = PatternFill("solid", start_color=LGRAY)
        lc.alignment = Alignment(vertical="center", horizontal="left")

        vc.font = Font(name="Arial", size=11, bold=True)
        vc.fill = PatternFill("solid", start_color=LGRAY)
        vc.alignment = Alignment(vertical="center", horizontal="center")
        if fmt:
            vc.number_format = fmt   # ← real Excel number format, not a string

    borders(ws, start_row, start_row + len(items), 1, 2)
    ws.column_dimensions['A'].width = 28
    ws.column_dimensions['B'].width = 20


# ══════════════════════════════════════════════════════════════════
# TASK 1 — METER READING
# ══════════════════════════════════════════════════════════════════
def task_meter_reading(wb, raw_df):
    print("  → Processing Meter Reading...")
    df = raw_df.copy()

    df['Prev_Reading'] = pd.to_numeric(df['Prev_Reading'], errors='coerce').fillna(0)
    df['Curr_Reading'] = pd.to_numeric(df['Curr_Reading'], errors='coerce').fillna(0)
    df['Avg_6M_Units'] = pd.to_numeric(df['Avg_6M_Units'], errors='coerce').fillna(0)
    df['Units_Consumed'] = df['Curr_Reading'] - df['Prev_Reading']

    def flag(row):
        u, avg = row['Units_Consumed'], row['Avg_6M_Units']
        if u < 0:   return 'IMPLAUSIBLE'
        if u == 0:  return 'INACTIVE/MINIMUM'
        if avg > 0 and abs(u - avg) / avg > 0.5: return 'HIGH VARIANCE'
        return 'NORMAL'

    df['Reading_Flag']    = df.apply(flag, axis=1)
    df['Action_Required'] = df['Reading_Flag'].map({
        'IMPLAUSIBLE':      'Request Check Reading from MIO',
        'INACTIVE/MINIMUM': 'Flag to Line Manager — possible bypass',
        'HIGH VARIANCE':    'Validate with field team before posting',
        'NORMAL':           'Clear for billing'
    })

    flag_bg = {
        'IMPLAUSIBLE':      LRED,
        'INACTIVE/MINIMUM': LBLUE,
        'HIGH VARIANCE':    YELLOW,
        'NORMAL':           None
    }
    row_bgs = [flag_bg.get(df.iloc[i]['Reading_Flag']) for i in range(len(df))]

    col_fmts = {
        'Prev_Reading':   FMT_NUM,
        'Curr_Reading':   FMT_NUM,
        'Avg_6M_Units':   FMT_NUM,
        'Units_Consumed': FMT_NUM,
    }

    ws = wb.create_sheet("1. Meter Reading Report")
    title(ws, "DPO METER READING — EXCEPTION REPORT", len(df.columns), BLUE)
    write_df(ws, df, start_row=4, hdr_bg=BLUE, col_fmts=col_fmts, row_bgs=row_bgs)

    counts = df['Reading_Flag'].value_counts()
    summary_block(ws, [
        ("Total Consumers",       len(df),                                    FMT_INT),
        ("Implausible Readings",  int(counts.get('IMPLAUSIBLE', 0)),          FMT_INT),
        ("High Variance",         int(counts.get('HIGH VARIANCE', 0)),        FMT_INT),
        ("Inactive / Minimum",    int(counts.get('INACTIVE/MINIMUM', 0)),     FMT_INT),
        ("Normal / Cleared",      int(counts.get('NORMAL', 0)),               FMT_INT),
    ], start_row=len(df) + 7, bg_title=BLUE, title_text="METER READING SUMMARY")

    print(f"     Implausible: {counts.get('IMPLAUSIBLE',0)}  |  High Variance: {counts.get('HIGH VARIANCE',0)}  |  Inactive: {counts.get('INACTIVE/MINIMUM',0)}")
    return df


# ══════════════════════════════════════════════════════════════════
# TASK 2 — RECOVERY ANALYSIS
# ══════════════════════════════════════════════════════════════════
def task_recovery(wb, raw_df):
    print("  → Processing Recovery...")
    df = raw_df.copy()

    df['Bill_Amount_PKR'] = pd.to_numeric(df['Bill_Amount_PKR'], errors='coerce').fillna(0)
    df['Arrears_PKR']     = pd.to_numeric(df['Arrears_PKR'],     errors='coerce').fillna(0)
    df['Amount_Paid']     = pd.to_numeric(df['Amount_Paid'],     errors='coerce').fillna(0)

    df['Total_Outstanding'] = df['Bill_Amount_PKR'] + df['Arrears_PKR']
    # Recovery_Pct stays as a float (0.0–1.0) — Excel formats it as % via number_format
    df['Recovery_Pct']      = (df['Amount_Paid'] /
                                df['Total_Outstanding'].replace(0, np.nan)).fillna(0)

    def dc_status(pct):
        if pct < 0.5:   return 'DC Candidate'
        if pct < 0.8:   return 'Warning — Send Notice'
        return 'Cleared'

    df['DC_Status']       = df['Recovery_Pct'].apply(dc_status)
    df['Action_Required'] = df['DC_Status'].map({
        'DC Candidate':          'Add to DC Schedule — prepare disconnection',
        'Warning — Send Notice': 'Send payment reminder notice',
        'Cleared':               'No action required'
    })

    status_bg = {
        'DC Candidate':          LRED,
        'Warning — Send Notice': YELLOW,
        'Cleared':               LGREEN,
    }
    row_bgs = [status_bg.get(df.iloc[i]['DC_Status']) for i in range(len(df))]

    col_fmts = {
        'Bill_Amount_PKR':  FMT_NUM,
        'Arrears_PKR':      FMT_NUM,
        'Total_Outstanding':FMT_NUM,
        'Amount_Paid':      FMT_NUM,
        'Recovery_Pct':     FMT_PCT,   # ← real % format, value stays 0.416
    }

    ws1 = wb.create_sheet("2. Recovery Analysis")
    title(ws1, "DPO RECOVERY — FULL ANALYSIS", len(df.columns), GREEN)
    write_df(ws1, df, start_row=4, hdr_bg=GREEN, col_fmts=col_fmts, row_bgs=row_bgs)

    total_out  = df['Total_Outstanding'].sum()
    total_paid = df['Amount_Paid'].sum()
    overall    = total_paid / total_out if total_out else 0

    summary_block(ws1, [
        ("Total Billed (PKR)",      df['Bill_Amount_PKR'].sum(),   FMT_NUM),
        ("Total Arrears (PKR)",     df['Arrears_PKR'].sum(),       FMT_NUM),
        ("Total Outstanding (PKR)", total_out,                     FMT_NUM),
        ("Total Collected (PKR)",   total_paid,                    FMT_NUM),
        ("Overall Recovery %",      overall,                       FMT_PCT),  # float → %
        ("DC Candidates",           int((df['DC_Status']=='DC Candidate').sum()), FMT_INT),
    ], start_row=len(df) + 7, bg_title=GREEN, title_text="RECOVERY SUMMARY")

    # DC Schedule sheet
    dc = (df[df['DC_Status'] == 'DC Candidate']
          .sort_values('Total_Outstanding', ascending=False).copy())

    ws2 = wb.create_sheet("3. DC Schedule")
    title(ws2, "DC SCHEDULE — DEFAULTERS (Sorted by Largest Outstanding)", len(dc.columns), RED)
    ws2['A2'] = (f"Generated: {datetime.today().strftime('%d-%b-%Y')}  |  "
                 f"DC Candidates: {len(dc)}  |  "
                 f"Total Outstanding: PKR {dc['Total_Outstanding'].sum():,.0f}")
    ws2['A2'].font = Font(name="Arial", italic=True, size=10, color="444444")
    write_df(ws2, dc, start_row=4, hdr_bg=RED,
             col_fmts=col_fmts,
             row_bgs=[LRED] * len(dc))

    print(f"     Overall Recovery: {overall:.1%}  |  DC Candidates: {len(dc)}")
    return df


# ══════════════════════════════════════════════════════════════════
# TASK 3 — SIR INSPECTION
# ══════════════════════════════════════════════════════════════════
def task_sir(wb, raw_df):
    print("  → Processing SIR Inspection...")
    df = raw_df.copy()

    df['Debit_Note_PKR'] = pd.to_numeric(df['Debit_Note_PKR'], errors='coerce').fillna(0)

    theft = ['Meter Bypass', 'Direct Connection', 'Meter Tampered']
    df['Is_Theft']    = df['Finding'].isin(theft)
    df['Case_Status'] = df.apply(
        lambda r: ('Closed'     if r['CA_Dispatched'] == 'Yes' and r['SAP_Updated'] == 'Yes'
                   else 'Pending-CA'  if r['CA_Dispatched'] == 'No'
                   else 'In Progress'), axis=1)
    df['Priority']    = df['Is_Theft'].map({True: 'HIGH — Theft Case', False: 'NORMAL'})

    row_bgs = []
    for _, r in df.iterrows():
        if r['Is_Theft']:               row_bgs.append(LRED)
        elif r['Case_Status'] == 'Pending-CA': row_bgs.append(YELLOW)
        else:                           row_bgs.append(None)

    col_fmts = {'Debit_Note_PKR': FMT_NUM}

    ws = wb.create_sheet("4. SIR Inspection Report")
    title(ws, "SIR METER INSPECTION REPORT", len(df.columns), PURPLE)
    write_df(ws, df, start_row=4, hdr_bg=PURPLE, col_fmts=col_fmts, row_bgs=row_bgs)

    closure_rate = (df['Case_Status'] == 'Closed').mean()   # float 0–1
    total_debit  = df['Debit_Note_PKR'].sum()

    summary_block(ws, [
        ("Total SIRs",              len(df),                                        FMT_INT),
        ("Closed",                  int((df['Case_Status']=='Closed').sum()),       FMT_INT),
        ("Pending CA Dispatch",     int((df['Case_Status']=='Pending-CA').sum()),   FMT_INT),
        ("In Progress",             int((df['Case_Status']=='In Progress').sum()),  FMT_INT),
        ("Closure Rate",            closure_rate,                                   FMT_PCT),  # float → %
        ("Theft Cases",             int(df['Is_Theft'].sum()),                      FMT_INT),
        ("Total Debit Notes (PKR)", total_debit,                                    FMT_NUM),
    ], start_row=len(df) + 7, bg_title=PURPLE, title_text="SIR SUMMARY")

    print(f"     Closure Rate: {closure_rate:.1%}  |  Pending CA: {(df['Case_Status']=='Pending-CA').sum()}  |  Theft: {df['Is_Theft'].sum()}")
    return df


# ══════════════════════════════════════════════════════════════════
# TASK 4 — LT LOSS CALCULATION
# ══════════════════════════════════════════════════════════════════
def task_lt_loss(wb, raw_df):
    print("  → Processing LT Loss...")
    df = raw_df.copy()

    df['Units_Received_Grid']    = pd.to_numeric(df['Units_Received_Grid'],    errors='coerce').fillna(0)
    df['Units_Billed_Consumers'] = pd.to_numeric(df['Units_Billed_Consumers'], errors='coerce').fillna(0)
    df['LT_Loss_Units'] = df['Units_Received_Grid'] - df['Units_Billed_Consumers']
    # LT_Loss_Pct stays as float (0.0–1.0) — formatted as % in Excel
    df['LT_Loss_Pct']   = (df['LT_Loss_Units'] /
                            df['Units_Received_Grid'].replace(0, np.nan)).fillna(0)

    def cat(pct):
        if pct > 0.25: return 'CRITICAL'
        if pct > 0.15: return 'HIGH'
        if pct > 0.08: return 'MEDIUM'
        return 'LOW'

    df['Loss_Category']   = df['LT_Loss_Pct'].apply(cat)
    df['Action_Required'] = df['Loss_Category'].map({
        'CRITICAL': 'Immediate Field Audit + SIR Drive',
        'HIGH':     'Schedule SIR Drive this month',
        'MEDIUM':   'Monitor weekly',
        'LOW':      'No action required'
    })

    # trend vs previous month per IBC
    df2 = df.sort_values(['IBC_Code', 'Month']).copy()
    df2['Prev_Loss_Pct'] = df2.groupby('IBC_Code')['LT_Loss_Pct'].shift(1)
    df2['Trend'] = df2.apply(
        lambda r: ('Worsening' if pd.notna(r['Prev_Loss_Pct']) and r['LT_Loss_Pct'] > r['Prev_Loss_Pct']
                   else 'Improving' if pd.notna(r['Prev_Loss_Pct']) and r['LT_Loss_Pct'] < r['Prev_Loss_Pct']
                   else 'Stable'), axis=1)

    ranked = df2.sort_values('LT_Loss_Pct', ascending=False).copy()
    # drop the helper column used only for trend calc
    ranked = ranked.drop(columns=['Prev_Loss_Pct'], errors='ignore')

    cat_bg = {'CRITICAL': LRED, 'HIGH': LORANGE, 'MEDIUM': YELLOW, 'LOW': LGREEN}
    row_bgs = [cat_bg.get(ranked.iloc[i]['Loss_Category'], LGRAY) for i in range(len(ranked))]

    col_fmts = {
        'Units_Received_Grid':    FMT_NUM,
        'Units_Billed_Consumers': FMT_NUM,
        'LT_Loss_Units':          FMT_NUM,
        'LT_Loss_Pct':            FMT_PCT2,  # float → 20.76% (2 decimals)
    }

    ws = wb.create_sheet("5. LT Loss Calculation")
    title(ws, "LT LOSS CALCULATION — MONTHLY REPORT (Ranked by Loss %)", len(ranked.columns), ORANGE)
    write_df(ws, ranked, start_row=4, hdr_bg=ORANGE, col_fmts=col_fmts, row_bgs=row_bgs)

    avg_loss   = df['LT_Loss_Pct'].mean()   # float
    total_loss = df['LT_Loss_Units'].sum()

    summary_block(ws, [
        ("Total Units Received",  float(df['Units_Received_Grid'].sum()),    FMT_NUM),
        ("Total Units Billed",    float(df['Units_Billed_Consumers'].sum()), FMT_NUM),
        ("Total Loss Units",      float(total_loss),                         FMT_NUM),
        ("Average Loss %",        avg_loss,                                  FMT_PCT2),  # float → %
        ("Critical IBCs",         int((df['Loss_Category']=='CRITICAL').sum()), FMT_INT),
        ("High IBCs",             int((df['Loss_Category']=='HIGH').sum()),     FMT_INT),
        ("Worsening Trend IBCs",  int((df2['Trend']=='Worsening').sum()),       FMT_INT),
    ], start_row=len(ranked) + 7, bg_title=ORANGE, title_text="LT LOSS SUMMARY")

    print(f"     Avg LT Loss: {avg_loss:.2%}  |  Critical: {(df['Loss_Category']=='CRITICAL').sum()}")
    return df


# ══════════════════════════════════════════════════════════════════
# TASK 5 — DASHBOARD
# ══════════════════════════════════════════════════════════════════
def task_dashboard(wb, mr_df, rec_df, sir_df, lt_df):
    print("  → Building Dashboard...")

    ws = wb.create_sheet("DASHBOARD", 0)
    ws.sheet_view.showGridLines = False
    title(ws, f"K-ELECTRIC DPO DAILY DASHBOARD — {datetime.today().strftime('%d %B %Y')}", 4, BLUE)
    ws.row_dimensions[1].height = 35

    total_out  = rec_df['Total_Outstanding'].sum()
    total_paid = rec_df['Amount_Paid'].sum()
    overall_rec = total_paid / total_out if total_out else 0
    closure_rate = (sir_df['Case_Status'] == 'Closed').mean()
    avg_lt_loss  = lt_df['LT_Loss_Pct'].mean()

    # sections: (heading, colour, [(label, value, format), ...])
    sections = [
        ("METER READING", BLUE, [
            ("Total Consumers",     len(mr_df),                                      FMT_INT),
            ("Implausible Readings",(mr_df['Reading_Flag']=='IMPLAUSIBLE').sum(),     FMT_INT),
            ("High Variance",       (mr_df['Reading_Flag']=='HIGH VARIANCE').sum(),   FMT_INT),
            ("Inactive / Minimum",  (mr_df['Reading_Flag']=='INACTIVE/MINIMUM').sum(),FMT_INT),
            ("Normal / Cleared",    (mr_df['Reading_Flag']=='NORMAL').sum(),           FMT_INT),
        ]),
        ("RECOVERY", GREEN, [
            ("Total Consumers",       len(rec_df),                                    FMT_INT),
            ("Total Outstanding PKR", total_out,                                      FMT_NUM),
            ("Total Collected PKR",   total_paid,                                     FMT_NUM),
            ("Overall Recovery %",    overall_rec,                                    FMT_PCT),
            ("DC Candidates",         int((rec_df['DC_Status']=='DC Candidate').sum()),FMT_INT),
        ]),
        ("SIR INSPECTION", PURPLE, [
            ("Total SIRs",          len(sir_df),                                      FMT_INT),
            ("Closed",              int((sir_df['Case_Status']=='Closed').sum()),      FMT_INT),
            ("Pending CA Dispatch", int((sir_df['Case_Status']=='Pending-CA').sum()), FMT_INT),
            ("Closure Rate %",      closure_rate,                                     FMT_PCT),
            ("Theft Cases",         int(sir_df['Is_Theft'].sum()),                    FMT_INT),
        ]),
        ("LT LOSS", ORANGE, [
            ("IBCs Analyzed",       len(lt_df),                                       FMT_INT),
            ("Avg LT Loss %",       avg_lt_loss,                                      FMT_PCT2),
            ("Critical IBCs",       int((lt_df['Loss_Category']=='CRITICAL').sum()),   FMT_INT),
            ("High Loss IBCs",      int((lt_df['Loss_Category']=='HIGH').sum()),        FMT_INT),
            ("Total Loss Units",    float(lt_df['LT_Loss_Units'].sum()),               FMT_NUM),
        ]),
    ]

    row = 3
    for sec_name, color, items in sections:
        c = ws.cell(row=row, column=1, value=sec_name)
        hdr(c, bg=color, sz=12)
        ws.merge_cells(f'A{row}:B{row}')
        ws.row_dimensions[row].height = 24
        row += 1
        for label, val, fmt in items:
            lc = ws.cell(row=row, column=1, value=label)
            vc = ws.cell(row=row, column=2, value=val)
            lc.font = Font(name="Arial", size=10, bold=True)
            lc.fill = PatternFill("solid", start_color=LGRAY)
            lc.alignment = Alignment(vertical="center", horizontal="left")
            vc.font = Font(name="Arial", size=11, bold=True, color=color)
            vc.fill = PatternFill("solid", start_color=LGRAY)
            vc.alignment = Alignment(vertical="center", horizontal="center")
            vc.number_format = fmt   # ← real Excel format on real numbers
            ws.row_dimensions[row].height = 20
            row += 1
        borders(ws, row - len(items) - 1, row - 1, 1, 2)
        row += 1

    ws.column_dimensions['A'].width = 28
    ws.column_dimensions['B'].width = 20


# ══════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════
def main():
    if not os.path.exists(INPUT_FILE):
        print(f"ERROR: '{INPUT_FILE}' not found. Place it in the same folder as this script.")
        sys.exit(1)

    print(f"\nK-Electric DPO Toolkit")
    print(f"Input : {INPUT_FILE}")
    print(f"Output: {OUTPUT_FILE}\n")

    print("Reading input file...")
    sheets = pd.read_excel(INPUT_FILE, sheet_name=None, header=1)
    for name in sheets:
        print(f"  '{name}' — {len(sheets[name])} rows")

    wb_out = Workbook()
    wb_out.remove(wb_out.active)

    print("\nProcessing tasks...")
    mr_df  = task_meter_reading(wb_out, sheets.get('Meter Reading',          pd.DataFrame()))
    rec_df = task_recovery(     wb_out, sheets.get('DPO Recovery',           pd.DataFrame()))
    sir_df = task_sir(          wb_out, sheets.get('SIR & Meter Inspection', pd.DataFrame()))
    lt_df  = task_lt_loss(      wb_out, sheets.get('LT Loss Calculation',    pd.DataFrame()))
    task_dashboard(wb_out, mr_df, rec_df, sir_df, lt_df)

    wb_out.save(OUTPUT_FILE)
    print(f"\nSaved: {OUTPUT_FILE}")
    print("Open in Excel — all numbers are real numbers, percentages are real %.\n")

if __name__ == "__main__":
    main()

: 